# Les thèmes de la presse de 1914

De quoi parlait *L'Humanité* en 1914, et comment l'actualité a-t-elle basculé l'été venu ? Sans aucune étiquette, BERTopic regroupe les passages proches et décrit chaque groupe par ses mots les plus caractéristiques.

1. Trouver les thèmes
2. Les ajuster
3. Les suivre dans le temps

Les *embeddings* des passages sont déjà calculés, avec Qwen3-Embedding-0.6B, et tronqués à 512 de ses 1 024 dimensions : le modèle est entraîné pour que ses premières dimensions suffisent, et la troncature est déjà faite. Ce notebook n'a pas besoin de GPU.

## Données

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/passages.parquet
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/embeddings-512.npy
%pip install -q "bertopic<1"

In [ ]:
import numpy as np
import pandas as pd

passages = pd.read_parquet("passages.parquet")
embeddings = np.load("embeddings-512.npy").astype(np.float32)

# L'Humanité seulement (Le Figaro n'a que 24 numéros), et un échantillon de 20 000
# passages pour que le calcul tienne en quelques minutes.
humanite = passages[passages.newspaper == "L'Humanité"].sample(20_000, random_state=0)
textes = humanite.text.tolist()
dates = humanite.date.tolist()
vecteurs = embeddings[humanite.index]
print(len(textes), vecteurs.shape)

## 1. Trouver les thèmes

Créez un modèle `BERTopic` qui réduit les *embeddings* à 5 dimensions avec UMAP (de façon reproductible), décrit les thèmes avec un `CountVectorizer` sans mots vides (ils se retirent pour **décrire les thèmes**, une **méthode lexicale** : c-TF-IDF compte les mots de chaque thème ; jamais avant les *embeddings*, que le modèle calcule sur le texte entier), et ne garde que les thèmes d'au moins 50 passages. Le corpus est en français.

Entraînez-le avec `fit_transform` (sur les vecteurs déjà calculés), puis affichez `get_topic_info()` : combien de thèmes ? **Part hors thème** : quelle part des passages n'a pas de thème (thème $-1$) ?

**Lire** : parcourez les mots-clés des 15 plus grands thèmes, et deux passages représentatifs (`get_representative_docs`) de trois d'entre eux. Les textes sortent de l'OCR des numéros de 1914 : des mots déformés, et des annonces, des cotes et des réclames, qui forment leurs propres thèmes (la colonne `noise` de `humanite` donne la part des mots inconnus de chaque passage). Quels mots-clés ne disent rien de leur thème ? Notez-les : l'exercice 2 les ajoute aux mots vides.

**Documentation utile**

- [`BERTopic`](https://maartengr.github.io/BERTopic/api/bertopic.html)
- [`BERTopic.fit_transform`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.fit_transform)
- [`BERTopic.get_topic_info`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.get_topic_info)
- [`BERTopic.get_representative_docs`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.get_representative_docs)
- [Paramètres d'UMAP](https://umap-learn.readthedocs.io/en/latest/parameters.html)
- [`CountVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from spacy.lang.fr.stop_words import STOP_WORDS
from umap import UMAP

MOTS_VIDES = sorted(STOP_WORDS | {"qu", "mm", "mme"})

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0)` : une réduction à 5 dimensions, reproductible
- `CountVectorizer(stop_words=MOTS_VIDES, min_df=10)` : BERTopic compte les mots sur un document par thème (tous ses passages mis bout à bout), donc `min_df=10` ignore les mots présents dans moins de 10 *thèmes*, et non 10 passages
- `BERTopic(language="french", umap_model=..., vectorizer_model=..., min_topic_size=50)` : sans `language="french"`, BERTopic, qui reçoit ici des vecteurs déjà calculés (sans `embedding_model`), retire des textes, avant de compter les mots, tout caractère autre qu'une lettre ASCII, un chiffre ou une espace (accents, apostrophes, ponctuation) : « grève » devient « grve »
- `fit_transform(textes, vecteurs)` : le second argument évite de recalculer les *embeddings* ; renvoie les thèmes de chaque passage, puis des probabilités
- `get_topic_info()` : une ligne par thème, le thème −1 étant celui des passages hors thème ; `np.mean(np.array(themes) == -1)` : leur part
- `get_topic(theme)` : les mots-clés d'un thème et leurs poids, dont `[mot for mot, _ in ...]` garde les mots ; `get_representative_docs(theme)` : ses passages représentatifs

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
modele_themes = BERTopic(
  language="french",
  umap_model=UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0
  ),
  vectorizer_model=CountVectorizer(stop_words=MOTS_VIDES, min_df=10),
  min_topic_size=...,
)
themes, _ = modele_themes.fit_transform(
  ..., ...
)  # les textes, puis les vecteurs déjà calculés
infos = modele_themes.get_topic_info()
print(len(infos) - 1, "thèmes,", f"{np.mean(np.array(themes) == -1):.0%} hors thème")
infos.head(16)

for theme in infos.Topic[1:4]:
  print(theme, [mot for mot, _ in modele_themes.get_topic(theme)[:8]])
  for texte in modele_themes.get_representative_docs(...)[:2]:
    print("   ", texte[:200])
```

### Solution

In [ ]:
modele_themes = BERTopic(
  language="french",
  umap_model=UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0
  ),
  vectorizer_model=CountVectorizer(stop_words=MOTS_VIDES, min_df=10),
  min_topic_size=50,
)
themes, _ = modele_themes.fit_transform(textes, vecteurs)
infos = modele_themes.get_topic_info()
print(len(infos) - 1, "thèmes,", f"{np.mean(np.array(themes) == -1):.0%} hors thème")
infos.head(16)

In [ ]:
for theme in infos.Topic[1:4]:
  print(theme, [mot for mot, _ in modele_themes.get_topic(theme)[:8]])
  for texte in modele_themes.get_representative_docs(theme)[:2]:
    print("   ", texte[:200])

# Les chiffres de ces commentaires (tailles, nombres et numéros des thèmes) changent
# d'une installation à l'autre, avec les versions de numpy et de numba : cherchez ce
# qu'ils illustrent. Ici, le plus grand thème (3 228 passages) mêle récits du front et
# roman-feuilleton : son premier passage représentatif est un dialogue de roman, le
# second un récit de la bataille de l'Yser. Chez vous, cherchez parmi les grands thèmes
# (élargissez `infos.Topic[1:4]` au besoin) celui dont les passages représentatifs
# parlent de deux choses. Des mots-clés comme « bien », « été », « rien », « oui »,
# « fut », « faire » ou « contre » ne disent rien du thème. Les annonces (« fr, rue,
# paris, 50, 75 ») et les réclames ont leurs propres thèmes.

## 2. Ajuster les thèmes

- Relancez avec `min_topic_size=20`, puis `100` : combien de thèmes, quelle part hors thème ?
- **Stabilité** : relancez le modèle de départ avec une autre graine pour UMAP (`random_state=1`). Ses grands thèmes reviennent-ils ? Pour chacun des cinq plus grands thèmes de départ, cherchez le nouveau thème qui reçoit le plus de ses passages (`pd.crosstab`), et quelle part.
- Ajoutez aux mots vides les mots-clés qui ne disent rien, notés à l'exercice 1, et recalculez les mots-clés du modèle de départ (`update_topics`) : sans réentraîner, les thèmes restent les mêmes, seule leur description change.
- Repérez deux thèmes qui parlent de la même chose et fusionnez-les (`merge_topics`).
- **Part hors thème** : réaffectez enfin les passages hors thème à leur thème le plus proche (`reduce_outliers`), puis mettez à jour les descriptions (`update_topics`). La fusion vient avant : une fois des thèmes donnés à la main, comme ceux de la réaffectation, réduire le nombre de thèmes peut échouer, et BERTopic le rappelle par un avertissement. Lisez cinq passages réaffectés : sont-ils à leur place ?

**Documentation utile**

- [Réduire les hors-thème](https://maartengr.github.io/BERTopic/getting_started/outlier_reduction/outlier_reduction.html)
- [`BERTopic.update_topics`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.update_topics)
- [`BERTopic.merge_topics`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.merge_topics)
- [`pandas.crosstab`](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- relancer avec une autre taille : dans une boucle sur `(20, 100)`, un nouveau `BERTopic` de même configuration, avec `min_topic_size=taille` (sans écraser `modele_themes`, le modèle de départ)
- une autre graine : un nouveau `BERTopic` de même configuration, avec `UMAP(..., random_state=1)` et `min_topic_size=50`
- `pd.crosstab(np.array(themes), np.array(themes_graine_1))` : le nombre de passages de chaque thème de départ (en ligne) dans chaque nouveau thème (en colonne) ; `croisement.loc[theme]` : la ligne d'un thème ; `.drop(-1, errors="ignore").idxmax()` : le nouveau thème qui en reçoit le plus, le thème −1 mis à part
- `sorted(set(MOTS_VIDES) | {...})` : les mots vides, plus les vôtres ; `modele_themes.update_topics(textes, vectorizer_model=CountVectorizer(stop_words=MOTS_VIDES_PRESSE, min_df=10))` : sans `topics=`, les thèmes restent ceux de l'entraînement, seuls les mots-clés sont recalculés
- `modele_themes.merge_topics(textes, [a, b])` : `a` et `b` sont deux thèmes à fusionner en un seul, choisis en lisant les mots-clés de `get_topic_info()`. Les numéros de thèmes changent après une fusion : `merge_topics` renumérote tous les thèmes et ne renvoie rien (`None`), reprenez donc `themes = modele_themes.topics_` juste après
- `modele_themes.reduce_outliers(textes, themes, strategy="embeddings", embeddings=vecteurs)` renvoie les nouveaux thèmes des passages
- `modele_themes.update_topics(textes, topics=nouveaux_themes, vectorizer_model=...)` : recalcule les mots-clés après la réaffectation, avec vos mots vides
- `pd.Series(nouveaux_themes)[np.array(themes) == -1].sample(5, random_state=0)` : cinq passages qui étaient hors thème, avec leur nouveau thème ; `textes[i]` : le texte du passage `i`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
for taille in (20, 100):
  modele = BERTopic(
    language="french",
    umap_model=UMAP(
      n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0
    ),
    vectorizer_model=CountVectorizer(stop_words=MOTS_VIDES, min_df=10),
    min_topic_size=...,
  )
  trouves, _ = modele.fit_transform(textes, vecteurs)
  print(
    f"min_topic_size={taille} : {len(set(trouves)) - 1} thèmes, {np.mean(np.array(trouves) == -1):.0%} hors thème"
  )

modele_graine_1 = BERTopic(
  language="french",
  umap_model=UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=...
  ),
  vectorizer_model=CountVectorizer(stop_words=MOTS_VIDES, min_df=10),
  min_topic_size=50,
)
themes_graine_1, _ = modele_graine_1.fit_transform(textes, vecteurs)
croisement = pd.crosstab(np.array(themes), np.array(themes_graine_1))
for theme in infos.Topic[1:6]:  # les cinq plus grands thèmes de départ
  ligne = croisement.loc[theme]
  proche = ligne.drop(-1, errors="ignore").idxmax()
  mots = [mot for mot, _ in modele_graine_1.get_topic(proche)[:5]]
  print(theme, f"{ligne[proche] / ligne.sum():.0%} dans le thème {proche}", mots)

MOTS_VIDES_PRESSE = sorted(set(MOTS_VIDES) | {...})  # vos mots-clés qui ne disent rien
modele_themes.update_topics(
  textes, vectorizer_model=CountVectorizer(stop_words=..., min_df=10)
)

# Deux thèmes qui parlent de la même chose :
modele_themes.merge_topics(textes, [..., ...])
themes = modele_themes.topics_  # les numéros ont changé

nouveaux_themes = modele_themes.reduce_outliers(
  textes, themes, strategy=..., embeddings=vecteurs
)
modele_themes.update_topics(textes, topics=nouveaux_themes, vectorizer_model=...)
print(f"{np.mean(np.array(nouveaux_themes) == -1):.0%} hors thème après réaffectation")
reaffectes = pd.Series(nouveaux_themes)[np.array(themes) == -1].sample(
  5, random_state=0
)
for i, theme in reaffectes.items():
  print(theme, [mot for mot, _ in modele_themes.get_topic(theme)[:5]])
  print("   ", textes[i][:200])
themes = nouveaux_themes
```

### Solution

In [ ]:
for taille in (20, 100):
  modele = BERTopic(
    language="french",
    umap_model=UMAP(
      n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0
    ),
    vectorizer_model=CountVectorizer(stop_words=MOTS_VIDES, min_df=10),
    min_topic_size=taille,
  )
  trouves, _ = modele.fit_transform(textes, vecteurs)
  print(
    f"min_topic_size={taille} : {len(set(trouves)) - 1} thèmes, {np.mean(np.array(trouves) == -1):.0%} hors thème"
  )

# Stabilité : le modèle de départ, avec une autre graine pour UMAP.
modele_graine_1 = BERTopic(
  language="french",
  umap_model=UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=1
  ),
  vectorizer_model=CountVectorizer(stop_words=MOTS_VIDES, min_df=10),
  min_topic_size=50,
)
themes_graine_1, _ = modele_graine_1.fit_transform(textes, vecteurs)
print(
  f"graine 1 : {len(set(themes_graine_1)) - 1} thèmes, {np.mean(np.array(themes_graine_1) == -1):.0%} hors thème"
)
croisement = pd.crosstab(np.array(themes), np.array(themes_graine_1))
for theme in infos.Topic[1:6]:  # les cinq plus grands thèmes de départ
  ligne = croisement.loc[theme]
  proche = ligne.drop(-1, errors="ignore").idxmax()
  mots = [mot for mot, _ in modele_graine_1.get_topic(proche)[:5]]
  print(theme, f"{ligne[proche] / ligne.sum():.0%} dans le thème {proche}", mots)

# Ici, avec la graine 1 : 44 thèmes, 38 % hors thème (d'autres versions de numpy et de
# numba donnent d'autres chiffres). À chercher : les thèmes nets reviennent presque
# entiers (ici faits divers, réunions, sport : 94 à 99 % de leurs passages dans un même
# thème ; les grèves à 58 %), et un thème qui se disperse entre de nouveaux thèmes et
# le hors-thème était un mélange. Ici, le plus grand se défait : 32 % de ses passages
# vont à un thème de roman (« bien, yeux, rien, tête »), 35 % hors thème. Si aucun
# grand thème ne se défait chez vous, ils tiennent avec cette graine : la stabilité se
# juge sur deux ou trois graines (2, 3).

In [ ]:
# Les mots-clés qui ne disent rien, repérés à l'exercice 1.
MOTS_VIDES_PRESSE = sorted(
  set(MOTS_VIDES) | {"été", "bien", "rien", "oui", "non", "fut", "faire", "contre"}
)
modele_themes.update_topics(
  textes, vectorizer_model=CountVectorizer(stop_words=MOTS_VIDES_PRESSE, min_df=10)
)
for theme in infos.Topic[1:4]:
  print(theme, [mot for mot, _ in modele_themes.get_topic(theme)[:8]])

# Ici, sans « bien », « été », « rien » ni « oui », les mots-clés du thème mêlé sont
# « allemands, yeux, hommes, troupes, ennemi, nuit » : la guerre et le roman, toujours
# mêlés. Les thèmes, eux, n'ont pas bougé : seule leur description a changé.

In [ ]:
# Ici, deux thèmes parlent des législatives d'avril-mai 1914 : les candidats d'un côté,
# les tableaux de résultats, abrégés, de l'autre. Leurs numéros changent avec les
# versions des bibliothèques : on les retrouve par leur mot-clé commun, « voix ».
elections = [
  theme
  for theme in modele_themes.get_topics()
  if "voix" in [mot for mot, _ in modele_themes.get_topic(theme)]
]
for theme in elections:
  print(theme, [mot for mot, _ in modele_themes.get_topic(theme)[:8]])

modele_themes.merge_topics(textes, elections)
themes = modele_themes.topics_  # la fusion a renuméroté les thèmes

# La réaffectation vient en dernier : l'avertissement de BERTopic le rappelle.
nouveaux_themes = modele_themes.reduce_outliers(
  textes, themes, strategy="embeddings", embeddings=vecteurs
)
modele_themes.update_topics(
  textes,
  topics=nouveaux_themes,
  vectorizer_model=CountVectorizer(stop_words=MOTS_VIDES_PRESSE, min_df=10),
)
print(f"{np.mean(np.array(nouveaux_themes) == -1):.0%} hors thème après réaffectation")
reaffectes = pd.Series(nouveaux_themes)[np.array(themes) == -1].sample(
  5, random_state=0
)
for i, theme in reaffectes.items():
  print(theme, [mot for mot, _ in modele_themes.get_topic(theme)[:5]])
  print("   ", textes[i][:200])
themes = nouveaux_themes

# Sur ces cinq passages, celui sur le Sénat est à sa place ; les autres sont à lire en
# entier : deux, dont un extrait de roman, rejoignent le grand thème mêlé de l'exercice 1.
# Réaffecter range chaque passage quelque part, pas forcément au bon endroit.

## 3. Suivre les thèmes dans le temps

`modele_themes.topics_over_time` compte les passages de chaque thème par semaine ; `modele_themes.visualize_topics_over_time` les trace. Affichez les thèmes qui parlent de la guerre, de Jaurès et du procès Caillaux : écrivez une fonction `theme_avec(mot)` qui renvoie le thème dont la plus grande part des passages contient `mot`.

Avant de tracer : le journal publie-t-il autant de passages chaque mois ? Comptez ceux de l'échantillon, mois par mois. Tracez ensuite, plutôt que le nombre de passages de chaque thème, leur **part** dans les passages de la semaine. Que se passe-t-il entre juillet et août 1914 ?

**Documentation utile**

- [Les thèmes dans le temps](https://maartengr.github.io/BERTopic/getting_started/topicsovertime/topicsovertime.html)
- [`BERTopic.topics_over_time`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.topics_over_time)
- [`BERTopic.visualize_topics_over_time`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.visualize_topics_over_time)
- [`Series.idxmax`](https://pandas.pydata.org/docs/reference/api/pandas.Series.idxmax.html)
- [`SeriesGroupBy.transform`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.SeriesGroupBy.transform.html)
- [`DataFrame.assign`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.assign.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `modele_themes.topics_over_time(textes, dates, nr_bins=52)` : 52 intervalles, une semaine chacun sur un an ; un tableau d'une ligne par thème et par semaine, avec le thème (`Topic`), son nombre de passages (`Frequency`) et la semaine (`Timestamp`)
- `theme_avec(mot)` : parmi les passages qui contiennent le mot (`mot in texte.lower()`), le nombre de passages de chaque thème (`pd.Series(themes)[masque].value_counts()`), divisé par la taille totale de ce thème (`pd.Series(themes).value_counts()`), sans le thème −1 (`.drop(-1, errors="ignore")`)
- `Series.idxmax()` : l'étiquette de la plus grande valeur, ici le numéro du thème ; essayez `"guerre"`, `"jaurès"` et `"caillaux"`
- `humanite.groupby(humanite.date.dt.month).size()` : le nombre de passages de chaque mois
- `themes_dans_le_temps.groupby("Timestamp").Frequency.transform("sum")` : pour chaque ligne, le nombre de passages de sa semaine ; `themes_dans_le_temps.assign(Frequency=themes_dans_le_temps.Frequency / totaux)` : le même tableau, avec des parts
- `modele_themes.visualize_topics_over_time(parts, topics=choisis)` : `topics=` est la liste des thèmes à tracer. Son option `normalize_frequency=True` ne règle pas la question : elle ramène chaque courbe à sa propre échelle, pas au volume du journal

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
themes_dans_le_temps = modele_themes.topics_over_time(textes, dates, nr_bins=...)


def theme_avec(mot: str) -> int:
  mentions = pd.Series(themes)[
    [mot in texte.lower() for texte in textes]
  ].value_counts()
  part = (mentions / pd.Series(themes).value_counts()).drop(-1, errors="ignore")
  return int(part.idxmax())


choisis = [theme_avec(mot) for mot in ("guerre", "jaurès", ...)]
print(choisis)
print(humanite.groupby(humanite.date.dt.month).size())  # passages par mois
totaux = themes_dans_le_temps.groupby("Timestamp").Frequency.transform(...)
parts = themes_dans_le_temps.assign(Frequency=themes_dans_le_temps.Frequency / totaux)
modele_themes.visualize_topics_over_time(..., topics=...)
```

### Solution

In [ ]:
themes_dans_le_temps = modele_themes.topics_over_time(textes, dates, nr_bins=52)


def theme_avec(mot: str) -> int:
  mentions = pd.Series(themes)[
    [mot in texte.lower() for texte in textes]
  ].value_counts()
  part = (mentions / pd.Series(themes).value_counts()).drop(-1, errors="ignore")
  return int(part.idxmax())


choisis = [theme_avec(mot) for mot in ("guerre", "jaurès", "caillaux")]
print(choisis)
print(humanite.groupby(humanite.date.dt.month).size())  # passages par mois
totaux = themes_dans_le_temps.groupby("Timestamp").Frequency.transform("sum")
parts = themes_dans_le_temps.assign(Frequency=themes_dans_le_temps.Frequency / totaux)
modele_themes.visualize_topics_over_time(parts, topics=choisis)

# L'échantillon compte 2 400 passages en juillet, 872 en août : le journal paraît chaque
# jour, mais ses numéros raccourcissent. En nombre, le thème de la guerre passe de 77
# passages en juillet à 120 en août ; en part, de 3 % à 14 %. En mars, Mme Caillaux tue
# Gaston Calmette, le directeur du Figaro ; son procès occupe juillet (10 % des passages
# les semaines du 16 et du 23). Jaurès est assassiné le 31 juillet : 17 % des passages de
# la semaine du 30.